# Complete Reward Head Training

这个 notebook 使用已经预计算好的 encoder cache 来完成 reward head 训练、train/evaluation loss 曲线保存、attention head 训练，以及后续指标比较。默认不会重新跑 LLM encoder。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd()

# 如果你的 precompute cache 目录不是这两个路径，改这里即可。
TRAIN_CACHE = PROJECT_ROOT / "cache" / "train"
VAL_CACHE = PROJECT_ROOT / "cache" / "val"

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"

EPOCHS = 10
LR = 1e-3
ZETA = 4.0

# 完整 sweep 会训练全部 head；如果只想比较旧 reward head 和 attention，可改成 ["linear", "attention"]。
HEADS = ["linear", "mlp", "cnn", "gru", "attention"]
COMPARE_HEADS = ["linear", "attention", "coin_flip"]
BASELINE_TRIALS = 100

CHECKPOINT_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

def run(cmd):
    print("\n$ " + " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True)

print("Project:", PROJECT_ROOT)
print("Train cache:", TRAIN_CACHE)
print("Val cache:", VAL_CACHE)

## 1. 检查 cache

这里确认 `hidden_size.txt` 和 `shard_*.pt` 都存在。这个 notebook 只训练 head，不做 precompute。

In [ ]:
def inspect_cache(cache_dir):
    cache_dir = Path(cache_dir)
    hidden_file = cache_dir / "hidden_size.txt"
    shards = sorted(cache_dir.glob("shard_*.pt"))
    if not hidden_file.exists():
        raise FileNotFoundError(f"Missing {hidden_file}")
    if not shards:
        raise FileNotFoundError(f"No shard_*.pt files in {cache_dir}")
    hidden_size = hidden_file.read_text().strip()
    print(f"{cache_dir}: hidden_size={hidden_size}, shards={len(shards)}")

inspect_cache(TRAIN_CACHE)
inspect_cache(VAL_CACHE)

## 2. 训练 reward heads 并保存 loss 曲线

每个 head 会输出：

- `checkpoints/{head}_head.pt`
- `results/{head}_efficiency.json`
- `results/{head}_loss_history.json`
- `results/{head}_loss_curve.png`，包含 train loss 和 evaluation loss

In [ ]:
for head in HEADS:
    run([
        sys.executable, "train_from_cache.py",
        "--cache_dir", TRAIN_CACHE,
        "--val_cache_dir", VAL_CACHE,
        "--head", head,
        "--epochs", EPOCHS,
        "--lr", LR,
        "--zeta", ZETA,
        "--save_path", CHECKPOINT_DIR / f"{head}_head.pt",
        "--results_dir", RESULTS_DIR,
    ])

## 3. 查看 loss 曲线

In [ ]:
from IPython.display import Image, display

for head in HEADS:
    path = RESULTS_DIR / f"{head}_loss_curve.png"
    if path.exists():
        print(head)
        display(Image(filename=str(path)))

## 4. Step-level evaluation

使用验证集 cache 计算 step reward accuracy 和 Q-value ranking accuracy。

In [ ]:
for head in HEADS:
    run([
        sys.executable, "eval_step_metrics.py",
        "--cache_dir", VAL_CACHE,
        "--head", head,
        "--checkpoint", CHECKPOINT_DIR / f"{head}_head.pt",
        "--results_dir", RESULTS_DIR,
    ])

## 5. 可选：single-eval cache 评估

如果已经有 `cache/single_eval`，这里会继续跑最终解答级别的准确率和 separation；没有就自动跳过。

In [ ]:
SINGLE_EVAL_CACHE = PROJECT_ROOT / "cache" / "single_eval"

if (SINGLE_EVAL_CACHE / "hidden_size.txt").exists() and list(SINGLE_EVAL_CACHE.glob("shard_*.pt")):
    for head in HEADS:
        run([
            sys.executable, "eval_single_from_cache.py",
            "--cache_dir", SINGLE_EVAL_CACHE,
            "--head", head,
            "--checkpoint", CHECKPOINT_DIR / f"{head}_head.pt",
            "--results_dir", RESULTS_DIR,
        ])
else:
    print(f"Skip single-eval: no cache found at {SINGLE_EVAL_CACHE}")

## 6. 随机掷硬币 baseline

这个 baseline 不训练模型，随机猜每一步或每条推理是否正确，并写入 `results/coin_flip_*.json`，用于最终表格比较。

In [ ]:
baseline_cmd = [
    sys.executable, "eval_coin_flip_baseline.py",
    "--val_cache_dir", VAL_CACHE,
    "--results_dir", RESULTS_DIR,
    "--trials", BASELINE_TRIALS,
]
if (SINGLE_EVAL_CACHE / "hidden_size.txt").exists() and list(SINGLE_EVAL_CACHE.glob("shard_*.pt")):
    baseline_cmd.extend(["--single_cache_dir", SINGLE_EVAL_CACHE])
run(baseline_cmd)

## 7. 汇总结果表

In [ ]:
run([
    sys.executable, "summarize_results.py",
    "--results_dir", RESULTS_DIR,
    "--out_csv", RESULTS_DIR / "summary.csv",
    "--out_md", RESULTS_DIR / "summary.md",
])

summary_md = RESULTS_DIR / "summary.md"
if summary_md.exists():
    print(summary_md.read_text())

## 8. Attention、旧 reward head 与随机 baseline 对比

In [ ]:
import csv
from IPython.display import Markdown, display

summary_csv = RESULTS_DIR / "summary.csv"
if summary_csv.exists():
    rows = list(csv.DictReader(summary_csv.open()))
    available = set(COMPARE_HEADS)
    cols = [
        "head",
        "final_train_loss",
        "final_eval_loss",
        "step_reward_accuracy",
        "step_reward_accuracy_std",
        "qvalue_ranking_accuracy",
        "qvalue_ranking_accuracy_std",
        "single_eval_accuracy",
        "single_eval_accuracy_std",
        "single_eval_separation",
        "single_eval_separation_std",
        "n_trainable_params",
        "total_train_time_sec",
    ]
    cols = [c for c in cols if rows and c in rows[0]]
    selected = [row for row in rows if row["head"] in available]
    if selected:
        table = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
        table.extend("| " + " | ".join(row.get(c, "") for c in cols) + " |" for row in selected)
        display(Markdown("\n".join(table)))
    else:
        print("No comparison heads found in summary.csv")
else:
    print("Run the summary cell first.")